<a href="https://colab.research.google.com/github/sarahibdah/Flyrank-ML-Internship-Sarah/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sarahibdah/Flyrank-ML-Internship-Sarah/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I'm going freestyle. My idea is "the half-life of content". The basic thought: every page has a life. It climbs, it peaks, and then it slowly fades. I want to use the daily warehouse data (about 78M rows over 17 months) to trace how pages actually live and die, and figure out what stage each page is in right now - still climbing, at its peak, or starting to fade. The lane guide calls this direction "Growth / Recovery / Momentum prediction" so I'm not inventing something with no guardrails. If it turns out the daily data is too noisy for this, my backup plan is to use the life-stage idea as one more feature in a normal refresh scoring queue (lane 2). I can still change my mind until end of week 4.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

The decision I want to improve is WHEN to refresh a page, not just which pages are already in trouble. Right now the rules fire after the traffic already dropped. I want a schedule instead: pages like this one usually peak around month 5, this page is at month 4, so refresh it next month.
Who acts on it: a content editor. They get a calendar of refreshes instead of an alarm.
Unit of analysis: one page over time (page per day in the warehouse, one row per page in the starter data).
Cost of a wrong call: if I say "refresh now" too early, the editor wastes hours on a page that was fine. If I say it too late, the traffic is already gone, which is what happens today anyway. So the whole point of this project is the timing.
Why ML at all: nobody can look at 78 million daily rows by hand, and a simple rule can hold one threshold but not the shape of a decay curve, which seems to be different for different content types and clients. That's the messy-but-real kind of pattern where a model makes sense.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/sarahibdah/Flyrank-ML-Internship-Sarah/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

base = (df["trend_direction"] == "down").mean()
print(f"Pages: {len(df):,} across {df['client_id'].nunique()} clients")
print(f"Base decline rate: {base:.1%}\n")

by_age = (df.groupby("age_tier_order")
            .agg(age_tier=("age_tier", "first"),
                 pages=("content_id", "count"),
                 decline_rate=("trend_direction", lambda s: s.eq("down").mean())))
print(by_age.to_string())

Pages: 30,000 across 32 clients
Base decline rate: 54.2%

               age_tier  pages  decline_rate
age_tier_order                              
3                 31-90    492      0.668699
4                91-180  11780      0.625552
5               181-365  11368      0.514866
6                  365+   6360      0.426258


The numbers surprised me. I expected old pages to be the declining ones, but it's backwards. Pages 31-90 days old decline at 66.9%, and pages over a year old only at 42.6% (base rate is 54.2%, 30,000 pages, 32 clients). I think this is survivorship: the weak old pages already died, so the old pages still around are the tough ones. Either way it means where a page is in its life carries signal that the current threshold rules ignore. One honest caveat: this table compares age groups at one moment in time, it doesn't follow the same pages through their life. That's exactly why I need the daily warehouse data - to turn this into real curves.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

What I can say at the end: observed and directional results, as decision support. Something like "pages that looked like this at this life stage went on to fade within N days X% of the time, tested on clients the model never saw". What I can't say: that I predicted Google's algorithm, or that refreshing a page causes it to recover (I only observe, I don't run experiments), or that this holds forever outside this snapshot.
The three traps I already know I have to watch: leakage (features must only come from before the prediction point, never after), survivorship (my age table above is partly explained by it), and the fact that different clients' daily history starts on different dates, so I can't use one global time window for everyone. If the curves end up too noisy to model honestly, I'll downgrade the claim and use life-stage as a feature in a lane 2 queue instead of overselling it.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.